In [1]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("jiashenliu/515k-hotel-reviews-data-in-europe")

print("Dataset Path:", path)
print("Files:", os.listdir(path))

100%|██████████| 45.1M/45.1M [00:00<00:00, 63.6MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/jiashenliu/515k-hotel-reviews-data-in-europe/versions/1
Files: ['Hotel_Reviews.csv']


In [2]:
import pandas as pd
import os

def find_sentiment(score):
    if score >= 8:
        return "Positive"
    elif score <= 5:
        return "Negative"
    else:
        return "Neutral"

csv_file_name = 'Hotel_Reviews.csv'
csv_file_path = os.path.join(path, csv_file_name)

reviews = pd.read_csv(csv_file_path)

reviews["Sentiment"] = reviews["Reviewer_Score"].apply(find_sentiment)
reviews["Sentiment"].value_counts()

,count
Sentiment,
Positive,335646
Neutral,149389
Negative,30703


In [3]:
print(os.listdir(path))

['Hotel_Reviews.csv']


In [4]:
reviews.head()

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng,Sentiment
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968,Negative
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968,Neutral
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968,Neutral
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968,Negative
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968,Neutral


In [5]:
reviews.info()
reviews.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515738 entries, 0 to 515737
Data columns (total 18 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   Hotel_Address                               515738 non-null  object 
 1   Additional_Number_of_Scoring                515738 non-null  int64  
 2   Review_Date                                 515738 non-null  object 
 3   Average_Score                               515738 non-null  float64
 4   Hotel_Name                                  515738 non-null  object 
 5   Reviewer_Nationality                        515738 non-null  object 
 6   Negative_Review                             515738 non-null  object 
 7   Review_Total_Negative_Word_Counts           515738 non-null  int64  
 8   Total_Number_of_Reviews                     515738 non-null  int64  
 9   Positive_Review                             515738 non-null  object 
 

,0
Hotel_Address,0
Additional_Number_of_Scoring,0
Review_Date,0
Average_Score,0
Hotel_Name,0
Reviewer_Nationality,0
Negative_Review,0
Review_Total_Negative_Word_Counts,0
Total_Number_of_Reviews,0
Positive_Review,0


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
if "Sentiment" not in reviews.columns:
    import pandas as pd
    reviews = pd.read_csv(csv_file_path)
    reviews["Sentiment"] = reviews["Reviewer_Score"].apply(find_sentiment)

x = reviews["Negative_Review"] + " " + reviews["Positive_Review"]
y = reviews["Sentiment"]

vectorizer = TfidfVectorizer(stop_words="english", max_features=10000, ngram_range=(1,2), min_df=5)
x = vectorizer.fit_transform(x)

In [7]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(x_train, y_train)
print("Model trained successfully!")

Model trained successfully!


In [9]:
prediction = model.predict(x_test)

In [10]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, prediction)
print("Accuracy :", round(accuracy * 100, 2), "%")

Accuracy : 74.17 %


In [11]:
from sklearn.metrics import classification_report
print(classification_report(y_test, prediction))

              precision    recall  f1-score   support

    Negative       0.65      0.19      0.29      6228
     Neutral       0.58      0.52      0.55     29937
    Positive       0.80      0.89      0.84     66983

    accuracy                           0.74    103148
   macro avg       0.68      0.53      0.56    103148
weighted avg       0.73      0.74      0.73    103148



In [12]:
my_review = [
    "The room was very clean and the staff were friendly."
]
review_vector = vectorizer.transform(my_review)
result = model.predict(review_vector)
print("Predicted Sentiment :", result[0])

Predicted Sentiment : Positive


In [13]:
my_review = [
    "The room was dirty and the service was terrible."
]
review_vector = vectorizer.transform(my_review)
result = model.predict(review_vector)
print("Predicted Sentiment :", result[0])

Predicted Sentiment : Negative


In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': [0.1, 0.5, 1.0, 1.5, 2.0]
}


grid_search = GridSearchCV(estimator=MultinomialNB(),
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=1)

grid_search.fit(x_train, y_train)
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation accuracy: ", round(grid_search.best_score_ * 100, 2), "%")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best parameters found:  {'alpha': 0.1}
Best cross-validation accuracy:  74.3 %


In [15]:
best_model = grid_search.best_estimator_

tuned_prediction = best_model.predict(x_test)

tuned_accuracy = accuracy_score(y_test, tuned_prediction)
print("Tuned Model Accuracy :", round(tuned_accuracy * 100, 2), "%")

print("\nClassification Report for Tuned Model:")
print(classification_report(y_test, tuned_prediction))

Tuned Model Accuracy : 74.21 %

Classification Report for Tuned Model:
              precision    recall  f1-score   support

    Negative       0.63      0.21      0.32      6228
     Neutral       0.58      0.52      0.55     29937
    Positive       0.80      0.89      0.84     66983

    accuracy                           0.74    103148
   macro avg       0.67      0.54      0.57    103148
weighted avg       0.73      0.74      0.73    103148



In [16]:
from sklearn.linear_model import LogisticRegression
log_reg_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
print("Training Logistic Regression model...")
log_reg_model.fit(x_train, y_train)
print("Logistic Regression model trained successfully!")

Training Logistic Regression model...
Logistic Regression model trained successfully!


In [17]:
from sklearn.metrics import accuracy_score, classification_report
log_reg_prediction = log_reg_model.predict(x_test)
log_reg_accuracy = accuracy_score(y_test, log_reg_prediction)
print("Logistic Regression Model Accuracy :", round(log_reg_accuracy * 100, 2), "%")
print("\nClassification Report for Logistic Regression Model:")
print(classification_report(y_test, log_reg_prediction))

Logistic Regression Model Accuracy : 75.55 %

Classification Report for Logistic Regression Model:
              precision    recall  f1-score   support

    Negative       0.64      0.30      0.41      6228
     Neutral       0.61      0.54      0.57     29937
    Positive       0.81      0.90      0.85     66983

    accuracy                           0.76    103148
   macro avg       0.69      0.58      0.61    103148
weighted avg       0.74      0.76      0.74    103148



In [18]:
my_review_positive = [
    "The room was very clean and the staff were friendly."
]
review_vector_positive = vectorizer.transform(my_review_positive)
result_positive = log_reg_model.predict(review_vector_positive)
print("Predicted Sentiment (Logistic Regression) for positive review:", result_positive[0])

my_review_negative = [
    "The room was dirty and the service was terrible."
]
review_vector_negative = vectorizer.transform(my_review_negative)
result_negative = log_reg_model.predict(review_vector_negative)
print("Predicted Sentiment (Logistic Regression) for negative review:", result_negative[0])

Predicted Sentiment (Logistic Regression) for positive review: Positive
Predicted Sentiment (Logistic Regression) for negative review: Negative
